# Resume small A/B — Qwen 14B NF4
Copy these three Code cells **in order into the existing Kaggle session**.
This creates a fresh source snapshot and result directory. No pip, unit tests,
CPU fixtures, model substitution, or repeated GPU stress.
Previously successful source/environment preparation and the real 4K+512 GPU
diagnostic are validated before reuse. Their original files remain unchanged.
Calibration reuse is recorded explicitly and does not approve generator quality.
Two genty modules, one seed, one round, both arms: 4 module trials.
Real upstream initialization still runs; no scripted model outputs or oracle copying.
The 60-minute budget can expire with failures; that is retained as evidence.
This notebook stops after small; it never launches the larger experiment.

Warm-prefix engineering retry: exact greedy initialization outputs are verified and imported.
Earlier timeout and cold initialization cost remain evidence. No generated tests or scoring outcomes are reused.

New development cohort: shared single-test-v1 output contract for both arms.
Initialization unchanged; all test generation and scoring are fresh.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, io, json, os, shutil, subprocess, sys, zipfile
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORKDIR=Path('/kaggle/working')/('Test4Py_upstream_'+STAMP)
ROOT=Path('/kaggle/working')/('UPSTREAM_NF4_COMPACT_'+STAMP)
OLD_SOURCE=Path('/kaggle/working/Test4Py_upstream_20261116T170358791891Z')
assert OLD_SOURCE.is_dir(), 'Copy these cells into the existing Kaggle session; no reinstall is needed.'
subprocess.run(['git','clone','--no-hardlinks',str(OLD_SOURCE),str(WORKDIR)],check=True)
BASE_COMMIT='7a97eb0a6c8fefc2e8154d4c0b4487b4ae2b0ea3'
if subprocess.check_output(['git','-C',str(WORKDIR),'rev-parse','HEAD'],text=True).strip()!=BASE_COMMIT:
    subprocess.run(['git','-C',str(WORKDIR),'fetch','--depth','1','origin',BASE_COMMIT],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'checkout','--detach','FETCH_HEAD'],check=True)
assert subprocess.check_output(['git','-C',str(WORKDIR),'rev-parse','HEAD'],text=True).strip()==BASE_COMMIT
HASHES={'ablation_runner.py': '2e139f4b8f083cb7d83157326ee39ed451d93fb5765a1facca0c55c95864154a', 'analyze.py': '50f8d6c65b26922b2df1a453317bc1d6bc1e279614c810404d7a389ffd3946dd', 'benchmark19.py': 'ad74c1b19dde7d6a09f3417040a79032eda8af18d8288f56feccc69f5fa840d1', 'benchmark_worker.py': 'c7a7573d05e5e54e7285ce75bac659ed5347c698426c54599a6bf2542b4264b1', 'bootstrap_kaggle.py': 'd432da528c8bbcf92647bfe09c734ac8c638e051a43a7eac0307cce42e44735c', 'bootstrap_modern.py': '3c591e4399847fa5d556e507ef8177fd5dbb9e2e8fd4049b5b3afdac9628fed5', 'check_environment.py': '7feb33cb0ebe2fa58ad87f6b2131ccbe766c8bf477c3c958d10b42af92668ef2', 'council/__init__.py': '329298564fa761413630b872e45b49d744051c50f47cc06d31edf3fcb3e8c773', 'council/critic.py': 'bcc350429209f03ef199d1f5d694d16c36b9165504dbc317bbbc1cadfc5468f9', 'council/engine.py': '655b0c407c72dc7939176b85c445f5ca9034eb4dbdbfb1c80077827ce3d2359b', 'council/generator.py': '15cceff340e99561a1b639b65ef26229cabbe4418abfcbb57ee1cb3c5aa982c3', 'council/runner.py': '2cbbdfadaa55740bc1d32835d21f4fd097433127ac28033c93fc69c573a2d9d4', 'council_multitarget.py': '950a2f86cf5fa5dcda70fc582471a9bc0b66873d7ffdd1d7e64029d28268668a', 'council_runner.py': 'ae2565edc07abc511f2c606f54ecdc3dc52b98c5254f4ccb0c43d150a780ecab', 'gpu_context_probe.py': 'f905795ce8a768030cc9a826670cc73944be0089a19ee81e48b586d70daa8311', 'kaggle_run.py': 'db922dd85c6358f72a26e027831809f87d544cd0af88e073bee0fec3531e4de1', 'notebooks/kaggle_multitarget_smoke_v4.ipynb': '788e0f93051164c82c339218b064550ae3cc845a751fb3a6a4e146601157dafe', 'paired_study.py': 'ff85c76f50fbd7f16a78f58ae40324951fb28c9c8ef089389d746b91a9b03941', 'pilot/__init__.py': '09b67cf5fee1772b0d9f5dacbbe62d7ab6b11d6e4cc28f8f4266f6b2e0e52464', 'pilot/attention.py': '2cbb0deb89364bbbb6908b7e88c59cdbe9f429bef4bc9eab2f4463bc8f92e855', 'pilot/backend.py': '4a367b4b5588c9fe688fe1486e0c334e55e9e112768ff2d147cf49d1336c9878', 'pilot/benchmark.py': 'dd0a38d738bd15793fcbc3d446f21ed2be27a0bfa163575f499cc7ed6700f374', 'pilot/common.py': 'ec56ff130444000176935bd9da14a50b397009aff15badce38b7c954049b7e89', 'pilot/console.py': '1f27112e62f7446ff50d8aed611e38d8bd99ba7442bd23d841cd9c4d324078fa', 'pilot/gpu_lease.py': '12505e17b0d451d1e8b89f680c8a77ce9423657ca4aa065094b0471f7bd19860', 'pilot/grounding.py': 'fe9be1949741591bcaffaacd337d75a4254ed1046f064b17f408c6463b624fa8', 'pilot/initialization_cache.py': '1e5caf4d3e4237308073a62b2ef8cfb3276798e7e20d0d48db96ed47fd3d64ec', 'pilot/instrumentation.py': '577bfde8f16fdfc2d12ec72ec3ebff7b6bcb163c1941e41f727053be8a83ef7c', 'pilot/measurement.py': '81f2d5e827f3eda5eed0cb64691645ed728bcd1fcf780d0caecfa98f725e3f54', 'pilot/model_profiles.py': '05a81e1e91f99f8fafa92d4c2e2f98f25a3e0e251c302ab46bf091d697d11083', 'pilot/module_mutation.py': 'f0d8ee86287f150f370ca25e4f92253ed663e01969ce7ef2f5966bea10d7e017', 'pilot/multitarget.py': '9f061ddaf7397a647a29a47cef30331e5aa1e8f23f6d307856619bf9085500bc', 'pilot/notebook.py': '5d94be4daa0eb2056ec9aa94d272c72915529795b215078c32d70cd3d51c42ff', 'pilot/output_contract.py': '091ac1891eaaf58fab5d129c64abf2715e3120ef2f18740cfcaa68607acb77a1', 'pilot/prescreen.py': '716a2e8f9b3d4ef913b5dbe5c48670673ffb184064ff094eb907cbb0a2ae19c2', 'pilot/probe.py': '367015a088a092aaec2021cc5f49796eae9cc6d15e7de16ed49ce4732d811c00', 'pilot/progress.py': 'b25db42a756435a67d665716a8ef6a29a4a1c008b7e06257e36833b504376675', 'pilot/quality.py': '92d31fb233a8102313368f251e83c120bd887ae81e423347bd9d39b303f82df7', 'pilot/reuse.py': '9e3603aec4448b713ce4c553c3221e4100217934c5d3ef2cc9b589416bf6cfd9', 'pilot/runtime_limits.py': 'a614bff522935b7f515440027d2d66aab2ce804da1d68fca52a5f2926d9db630', 'pilot/scheduling.py': '3b781e64f28872a5ea792d9c12eda47caa0b271a331c75191d58fc1a9a618861', 'pilot/scoring.py': '66b847795ce11523fbda9cd49c4347ac5d29be37fcddc3e18e07ed7c56a1dd5d', 'pilot/screen_trace.py': '77feb353b6f2952b91336c978481014d7e55bd1bccc633df57ce25cac6e6bcd2', 'pilot/setup.py': '9b1f1e579a6a0526f3e92813faa0e3b2a8d6c925a5f59aca22c56d69b52fb5a9', 'pilot/telemetry.py': 'de151bd6880595c98a3bfd2cbbc1e0b26d7a13079589df70340fe1857886150f', 'pilot/upstream_adapter.py': 'dfef842a9fe5e0b8db5054da2ee77b2e45273470ae2462379fb8f7c8367dcb17', 'pilot/validate.py': 'd1e57a63496b5985c182a56025c083d3778d299f3d3cf5ba136a8710b9b890f3', 'pilot.json': 'af87638ffa25424041c9b172e11aaa160e6efce34ffcb50dad7953d86187a210', 'pytest.ini': '0d6e34350f9e9a731b52183ee69d00334a8c560d849ccc9278b1ebfe48c73a9d', 'requirements-analysis.txt': '6fe163ca5631bab1ca772c8ea624297c2b3b5b86b40d9059e7ceebf6eb717510', 'requirements-modern.txt': '97c174f60ea4a808b2ed8cfb477bcc9188df1aedc65802012cbdbfce82657fa4', 'requirements-notebook.txt': '9847fe4dc03a9a4f1ca1752873d1ed97035c75ad21f699b68fcca811d366f879', 'requirements-paired.txt': 'e603a5074d7ccd0531e5c37ebb731a96c3d038fff9c8691af81e8a01de885cf6', 'research/benchmark/na-projects.json': '97e0a0a272046b858fa1e46b82f8b2b0afb3d81da3f0571d84edc3d0c58d7787', 'research/benchmark/projects-upstream.json': '54bfd0a005ae780525a6c8f5051f95a69671a40e989f0b2ddfba42c5ffc4d8c5', 'resume_paired.py': '6c5bd5e08d9e21482c0ea263ac56be60cfe2e7ae6a1874fd1f7db8476f1d4e1f', 'run.py': '340c51d7a90296e8f842ca7a2da7d8961499e5e620f5ab963667acf5e42dc928', 'run_research.py': 'aeea0208490def7517fb3f51d5354fdadd6cd4d9a97dcf5bd7c9ff643e65ae9e', 'scripts/check_upstream_fixture.py': '09503ddb82daf9533114ba28c1a8bfa8ad904b38c9cb2de9b97049e70db3881c', 'scripts/check_upstream_initialization.py': 'd72516f78e0c43814d5c742d0a6b7a414b7fbf94c0a52498209341484dbe1e29', 'scripts/kaggle_quick_oom_check.py': '92bde4593ceb0357a1a7b579008e7ff606e93892ba2f43faabb52d0c337fcab3', 'setup_context.py': '44e697a8e6963ba58e19ad080ad4dd80289420c28d4f73eaf45e32548c3f9613', 'smoke_pilot.py': '39ae9ee5e07bddbc5ddce481335a1c1a56f67d9b210215e65f9bdf7c266d3853', 'tests/test_attention.py': 'a9769aab4c9c8029b2ad0483add6397923ef17676c5f39dce299bffcccb020d1', 'tests/test_benchmark.py': 'fe51c6b1702d16e2dd2b24d13408fc752a5cd3f65ce3ac84b7211b5ffa9397a0', 'tests/test_council.py': 'a5a2a881fc6a02c85dfaada0919db6aadbe2c7d800738ad4c7a96629d9b23696', 'tests/test_gpu_lease.py': 'acd030148885a0a288197d52314ce4789eac48051a463196fe9d581eec85d71d', 'tests/test_measurement.py': 'e1f014ed54edb8dbfa4ecdb080725ebd01ef50bac6017b42cf473bc428aa2104', 'tests/test_multitarget.py': '3df432a208590398d39f3eb3b7f2305d35c13b375169ed65448ce79b2e22341c', 'tests/test_notebook_runtime.py': '7485c0b0fb55d16ba904ba4342f1c99fc33e771221950c024c36cd12d7c86f0b', 'tests/test_output_contract.py': '211d67e97663c526b78123da93bf3df87f15bba4c2e3e9a66a7ee02c3b048d22', 'tests/test_paired.py': '4cae46201aadafed6e1268bf4e80fd9d4d30df97516b4bdf7a941c8a93463842', 'tests/test_pilot.py': '2705fcd9bef011a5b3e0490483b43d600e1fc1b590bdac38c000de516bc59186', 'tests/test_prescreen.py': '0af81cea71e202fba5127a87243f48236a479641277f45f47588b3bb7d151833', 'tests/test_progress_counts.py': '2e5b64a726e08fe172c4adb2c2b1d23de261a581eb7cbacb477553ab3cab97ab', 'tests/test_quality.py': 'b42b9de6dd07835b2e92ad6c04d24fca7aa89e48f93cb6311c97ff1d9ac2a665', 'tests/test_research.py': '9cf673d2d2511c655efbdcffa14a7b39a44a47a08c6b5fd97c61a8fea82e522c', 'tests/test_reuse.py': '982692dfb1fe207b6550c0192f8c36d07e31bd9b66a69f436e6708834ea42b6b', 'worker.py': 'bb9a14655f9dbecddde4fa352e05895d31e61a8f9e1306b4c145665000676aae'}
PAYLOAD='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    assert set(archive.namelist())==set(HASHES)
    for name,expected in HASHES.items():
        data=archive.read(name)
        assert hashlib.sha256(data).hexdigest()==expected,name
        path=(WORKDIR/name).resolve()
        assert path.is_relative_to(WORKDIR.resolve())
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(data)
os.chdir(WORKDIR)
sys.path.insert(0,str(WORKDIR))
PROVENANCE=dict(base_commit=BASE_COMMIT,overlay_sha256=HASHES,source_kind='Git base plus embedded local changes; not claimed pushed')
print('Source:',WORKDIR,'Study:',ROOT)


In [ ]:
# Reuse successful preparation and calibrated memory; no installs/unit/CPU fixtures.
PYTHON='/tmp/test4py-modern/bin/python'
assert Path(PYTHON).is_file(), 'Prepared interpreter missing; use the same Kaggle session.'
LOG=Path('/kaggle/working')/('UPSTREAM_RESUME_'+STAMP+'.log')
FAILED_STEP=None
def run_step(arguments, env=None):
    global FAILED_STEP
    if FAILED_STEP is not None:
        raise RuntimeError('Previous step failed; later steps are blocked. Inspect: '+str(FAILED_STEP))
    with LOG.open('a') as log:
        proc=subprocess.Popen([PYTHON,*arguments],cwd=WORKDIR,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        code=proc.wait()
    if ROOT.exists():
        shutil.copy2(LOG,ROOT/'notebook-launcher.log')
    if code:
        FAILED_STEP=dict(arguments=arguments,exit_code=code)
        if ROOT.exists():
            (ROOT/'notebook-failure.json').write_text(json.dumps(FAILED_STEP,indent=2))
            for path in sorted(ROOT.glob('stress-gpu-*/probe.json')):
                evidence=json.loads(path.read_text())
                print(path, 'status:',evidence.get('status'),'error:',evidence.get('error'))
                for test in evidence.get('tests',[]):
                    print('Stress measurement:',test)
            from paired_study import make_bundle
            bundle=make_bundle(ROOT)
            from IPython.display import display,FileLink
            display(FileLink(str(bundle)))
        raise RuntimeError(f'Step exited {code}; retained artifacts at {ROOT}. Do not proceed to larger run.')
run_step(['paired_study.py','--action','plan','--output',str(ROOT),
          '--max-input-tokens','4096','--max-new-tokens','512',
          '--test-output-contract','single-test-v1'])
(ROOT/'notebook-source.json').write_text(json.dumps(PROVENANCE,indent=2))
run_step(['resume_paired.py','--previous','/kaggle/working/UPSTREAM_NF4_20261116T170358791891Z',
          '--diagnostic','/kaggle/working/NF4_14B_OOM_CHECK_20261116T172805400121Z',
          '--initialization-from','/kaggle/working/UPSTREAM_NF4_RESUME_20261116T174754946339Z',
          '--output',str(ROOT)])


In [ ]:
# REAL 14B inference. Two genty modules x baseline/improved, seed 42, one round.
# 60 minutes includes the embedding download and model loads. All outcomes retained.
run_step(['paired_study.py','--action','small','--output',str(ROOT),'--jobs','2'])
from IPython.display import display,FileLink,Markdown
display(Markdown((ROOT/'small-report.md').read_text()))
print(json.loads((ROOT/'small-measurement-summary.json').read_text()))
display(FileLink(str(ROOT)+'.zip'))
print('Larger experiment NOT launched. Send this archive for review.')
